
# P8 Figure and Table Source Assembly

This notebook freezes the manuscript-facing source tables for the current main-paper object set:

- Figures: `F1` through `F6`
- Tables: `T1` through `T3`

This batch does not rerun the analytical pipeline. It assembles source-ready objects from the frozen outputs of `P1` through `P7`.


In [ ]:

from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'artifacts' / 'tables' / 'p1_retained_sample_summary.csv').exists():
    ROOT = ROOT.parent

TABLES = ROOT / 'artifacts' / 'tables'
REPORTS = ROOT / 'artifacts' / 'reports'
RUNTIME = ROOT / 'artifacts' / 'runtime'

for path in [TABLES, REPORTS, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)

PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260320__P8__nogit'
EXPECTED_MAIN_FIGURES = ['F1', 'F2', 'F3', 'F4', 'F5', 'F6']
EXPECTED_MAIN_TABLES = ['T1', 'T2', 'T3']
EXPECTED_COUNTRIES = 94
EXPECTED_CAPACITY_COMMUNITIES = 6
EXPECTED_COUNT_ORIGINAL_COMMUNITIES = 6
EXPECTED_COUNT_MAINTEXT_CAMPS = 5
EXPECTED_MODES = 16

OBJECT_TITLES = {
    'F1': 'Workflow + retained scope',
    'F2': 'Raw vs row-normalized concentration/network contrast',
    'F3': 'Capacity-based camps',
    'F4': 'Null-model comparison',
    'F5': 'Count-based higher-order camps',
    'F6': 'Mode distinction audit',
    'T1': 'Retained scope / inclusion table',
    'T2': 'Camp summary and dominant-mode table',
    'T3': 'Null-model and robustness summary table',
}

CLAIM_SUPPORT = {
    'F1': 'context for C9-C10',
    'F2': 'C2',
    'F3': 'C1, C4',
    'F4': 'C3',
    'F5': 'C5, C7',
    'F6': 'C6',
    'T1': 'C9, C10',
    'T2': 'C1, C4, C5',
    'T3': 'C3, C8',
}

manifest = {
    'notebook': 'P8_Figure_Table_Source_Assembly.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'run_id': RUN_ID,
    'main_figures': EXPECTED_MAIN_FIGURES,
    'main_tables': EXPECTED_MAIN_TABLES,
    'object_titles': OBJECT_TITLES,
}
(RUNTIME / 'p8_source_assembly_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 240)

display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown('**Batch role:** freeze manuscript-facing figure/table source objects from prior batches only'))



## Load frozen upstream outputs


In [ ]:

p1_sample_summary = pd.read_csv(TABLES / 'p1_retained_sample_summary.csv')
p1_country_scope = pd.read_csv(TABLES / 'p1_country_scope_summary.csv')
p1_status_scope = pd.read_csv(TABLES / 'p1_status_scope_summary.csv')
p1_country_filter = pd.read_csv(TABLES / 'p1_country_filter_audit.csv')
p1_bin_audit = pd.read_csv(TABLES / 'p1_bin_assignment_audit.csv')

p2_diagnostics = pd.read_csv(TABLES / 'p2_concentration_diagnostics.csv')

p3_capacity_membership = pd.read_csv(TABLES / 'p3_capacity_community_membership.csv')
p3_capacity_profiles = pd.read_csv(TABLES / 'p3_capacity_dominant_mode_profiles.csv')
p3_count_membership = pd.read_csv(TABLES / 'p3_count_community_membership.csv')
p3_count_profiles = pd.read_csv(TABLES / 'p3_count_dominant_mode_profiles.csv')

p4_summary = pd.read_csv(TABLES / 'p4_null_model_summary.csv')
p4_draws_count = pd.read_csv(TABLES / 'p4_null_model_draws_count.csv')
p4_draws_capacity = pd.read_csv(TABLES / 'p4_null_model_draws_capacity.csv')

p5_classes = pd.read_csv(TABLES / 'p5_mode_distinction_classes.csv')

p6_crosswalk = pd.read_csv(TABLES / 'p6_count_community_to_camp_crosswalk.csv')

p7_binning = pd.read_csv(TABLES / 'p7_binning_stability_summary.csv')
p7_countryfilter = pd.read_csv(TABLES / 'p7_countryfilter_sensitivity_summary.csv')

summary_lookup = p1_sample_summary.set_index('metric')['value'].to_dict()



## Helper utilities


In [ ]:

def metric_value(metric, cast=None):
    value = summary_lookup[metric]
    if cast is None:
        return value
    return cast(value)


def file_label(path):
    return Path(path).name if isinstance(path, (str, Path)) else str(path)


def with_object_metadata(df, object_code, object_type, title, claim_support, source_batch):
    out = df.copy()
    out.insert(0, 'object_code', object_code)
    out.insert(1, 'object_type', object_type)
    out.insert(2, 'object_title', title)
    out.insert(3, 'claim_support', claim_support)
    out.insert(4, 'source_batch', source_batch)
    out.insert(5, 'run_id', RUN_ID)
    out.insert(6, 'pipeline_version', PIPELINE_VERSION)
    return out



## Assemble figure source tables


In [ ]:

# F1
workflow_rows = pd.DataFrame([
    {'display_order': 1, 'section': 'workflow', 'item_key': 'P1', 'item_label': 'Retained base tables', 'item_value': 'scope freeze + retained baseline', 'upstream_file': 'p1_retained_scope_report.md'},
    {'display_order': 2, 'section': 'workflow', 'item_key': 'P2', 'item_label': 'Core matrices', 'item_value': 'raw + row-normalized country-mode matrices', 'upstream_file': 'p2_normalization_report.md'},
    {'display_order': 3, 'section': 'workflow', 'item_key': 'P3', 'item_label': 'Camp detection', 'item_value': 'capacity main camps + count complementary camps', 'upstream_file': 'p3_camp_summary_report.md'},
    {'display_order': 4, 'section': 'workflow', 'item_key': 'P4', 'item_label': 'Null confirmation', 'item_value': 'degree-preserving null comparison', 'upstream_file': 'p4_null_model_report.md'},
    {'display_order': 5, 'section': 'workflow', 'item_key': 'P5', 'item_label': 'Mode distinction', 'item_value': 'discriminative vs background vs head-dominated modes', 'upstream_file': 'p5_mode_distinction_report.md'},
    {'display_order': 6, 'section': 'workflow', 'item_key': 'P6', 'item_label': 'Count merge audit', 'item_value': '6 original count communities -> 5 main-text camps', 'upstream_file': 'p6_count_merge_report.md'},
    {'display_order': 7, 'section': 'workflow', 'item_key': 'P7', 'item_label': 'Sensitivity suite', 'item_value': 'approved alternative binning + country-filter robustness', 'upstream_file': 'p7_sensitivity_report.md'},
])

scope_keys = [
    'dataset_release', 'source_sheet', 'source_scope', 'observational_unit', 'distributed_sheet_excluded',
    'retained_statuses', 'excluded_statuses', 'country_filter_rule', 'size_bins', 'bin_boundary_rule',
    'countries_before_country_filter', 'countries_retained', 'countries_removed_by_country_filter',
    'rows_after_status_and_capacity_filters', 'rows_retained', 'rows_removed_by_country_filter',
    'capacity_mw_retained', 'capacity_mw_removed_by_country_filter'
]
scope_rows = p1_sample_summary.loc[p1_sample_summary['metric'].isin(scope_keys)].copy()
scope_rows['display_order'] = np.arange(101, 101 + len(scope_rows))
scope_rows['section'] = 'retained_scope'
scope_rows['item_key'] = scope_rows['metric']
scope_rows['item_label'] = scope_rows['metric']
scope_rows['item_value'] = scope_rows['value'].astype(str)
scope_rows['upstream_file'] = 'p1_retained_sample_summary.csv'
scope_rows = scope_rows[['display_order', 'section', 'item_key', 'item_label', 'item_value', 'upstream_file']]

F1_source = pd.concat([workflow_rows, scope_rows], ignore_index=True)
F1_source = with_object_metadata(F1_source, 'F1', 'figure', OBJECT_TITLES['F1'], CLAIM_SUPPORT['F1'], 'P1-P7')
F1_source.to_csv(TABLES / 'p8_F1_source.csv', index=False)

# F2
F2_source = p2_diagnostics.copy()
F2_source['panel'] = F2_source['weight_kind'].map({'count': 'count_view', 'capacity_mw': 'capacity_view'})
F2_source = F2_source.sort_values(['weight_kind', 'matrix_variant']).reset_index(drop=True)
F2_source = with_object_metadata(F2_source, 'F2', 'figure', OBJECT_TITLES['F2'], CLAIM_SUPPORT['F2'], 'P2')
F2_source.to_csv(TABLES / 'p8_F2_source.csv', index=False)

# F3
capacity_profile_cols = [
    'community', 'countries', 'country_share', 'raw_total_weight', 'raw_weight_share',
    'graph_strength_sum', 'graph_strength_share', 'dominant_mode', 'dominant_mode_share',
    'second_mode', 'second_mode_share', 'top3_modes', 'top3_share',
    'dominant_status', 'dominant_tier', 'interpretable'
]
F3_source = p3_capacity_membership.merge(
    p3_capacity_profiles[capacity_profile_cols].rename(columns={
        'countries': 'community_countries',
        'country_share': 'community_country_share',
        'raw_total_weight': 'community_raw_total_weight',
        'raw_weight_share': 'community_raw_weight_share',
        'graph_strength_sum': 'community_graph_strength_sum',
        'graph_strength_share': 'community_graph_strength_share',
        'dominant_mode': 'community_dominant_mode_profile',
        'dominant_mode_share': 'community_dominant_mode_share',
        'second_mode': 'community_second_mode',
        'second_mode_share': 'community_second_mode_share',
        'top3_modes': 'community_top3_modes',
        'top3_share': 'community_top3_share',
        'dominant_status': 'community_dominant_status',
        'dominant_tier': 'community_dominant_tier',
        'interpretable': 'community_interpretable',
    }),
    on='community',
    how='left'
).sort_values(['community', 'graph_strength'], ascending=[True, False]).reset_index(drop=True)
F3_source = with_object_metadata(F3_source, 'F3', 'figure', OBJECT_TITLES['F3'], CLAIM_SUPPORT['F3'], 'P3')
F3_source.to_csv(TABLES / 'p8_F3_source.csv', index=False)

# F4
null_draws = pd.concat([
    p4_draws_count.assign(weight_kind='count'),
    p4_draws_capacity.assign(weight_kind='capacity'),
], ignore_index=True)
F4_source = null_draws.merge(p4_summary, on='weight_kind', how='left', suffixes=('', '_summary'))
F4_source['record_type'] = 'null_draw'
summary_rows = p4_summary.copy()
summary_rows['replicate_id'] = np.nan
summary_rows['seed'] = np.nan
summary_rows['rewire_success'] = np.nan
summary_rows['degree_sequence_preserved'] = np.nan
summary_rows['null_modularity'] = np.nan
summary_rows['record_type'] = 'summary'
F4_source = pd.concat([F4_source, summary_rows[F4_source.columns]], ignore_index=True)
F4_source = F4_source.sort_values(['weight_kind', 'record_type', 'replicate_id'], ascending=[True, False, True], na_position='last').reset_index(drop=True)
F4_source = with_object_metadata(F4_source, 'F4', 'figure', OBJECT_TITLES['F4'], CLAIM_SUPPORT['F4'], 'P4')
F4_source.to_csv(TABLES / 'p8_F4_source.csv', index=False)

# F5
camp_country_totals = p6_crosswalk.groupby('main_text_count_camp', as_index=False)['original_countries'].sum().rename(columns={'original_countries': 'main_text_camp_countries'})
F5_source = p3_count_membership.merge(
    p6_crosswalk.rename(columns={'original_count_community': 'community'}),
    on='community',
    how='left'
).merge(
    camp_country_totals,
    on='main_text_count_camp',
    how='left'
).sort_values(['main_text_count_camp', 'community', 'graph_strength'], ascending=[True, True, False]).reset_index(drop=True)
F5_source = with_object_metadata(F5_source, 'F5', 'figure', OBJECT_TITLES['F5'], CLAIM_SUPPORT['F5'], 'P3+P6')
F5_source.to_csv(TABLES / 'p8_F5_source.csv', index=False)

# F6
F6_source = p5_classes.sort_values(['shared_priority', 'mode'], ascending=[False, True]).reset_index(drop=True)
F6_source = with_object_metadata(F6_source, 'F6', 'figure', OBJECT_TITLES['F6'], CLAIM_SUPPORT['F6'], 'P5')
F6_source.to_csv(TABLES / 'p8_F6_source.csv', index=False)

display(Markdown('**Figure source files saved**'))
display(pd.DataFrame({'object_code': EXPECTED_MAIN_FIGURES, 'title': [OBJECT_TITLES[x] for x in EXPECTED_MAIN_FIGURES]}))



## Assemble table source tables


In [ ]:

# T1
T1_keep = [
    'dataset_release', 'source_sheet', 'source_scope', 'observational_unit', 'distributed_sheet_excluded',
    'retained_statuses', 'excluded_statuses', 'country_filter_rule', 'size_bins', 'bin_boundary_rule',
    'countries_before_country_filter', 'countries_retained', 'countries_removed_by_country_filter',
    'rows_after_status_and_capacity_filters', 'rows_retained', 'rows_removed_by_country_filter',
    'capacity_mw_retained', 'capacity_mw_removed_by_country_filter', 'minimum_retained_country_record_count',
]
T1_source = p1_sample_summary.loc[p1_sample_summary['metric'].isin(T1_keep)].copy().reset_index(drop=True)
T1_source['display_order'] = np.arange(1, len(T1_source) + 1)
T1_source = with_object_metadata(T1_source, 'T1', 'table', OBJECT_TITLES['T1'], CLAIM_SUPPORT['T1'], 'P1')
T1_source.to_csv(TABLES / 'p8_T1_source.csv', index=False)

# T2
capacity_t2 = p3_capacity_profiles[[
    'community', 'community_name_draft', 'countries', 'country_share', 'raw_total_weight', 'raw_weight_share',
    'dominant_mode', 'dominant_mode_share', 'second_mode', 'second_mode_share', 'top3_modes', 'top3_share',
    'dominant_status', 'dominant_tier', 'interpretable'
]].copy()
capacity_t2['lens'] = 'capacity_main'
capacity_t2['camp_id'] = capacity_t2['community']
capacity_t2['camp_name_draft'] = capacity_t2['community_name_draft']
capacity_t2['source_communities'] = capacity_t2['community'].astype(str)
capacity_t2['compression_layer'] = 'algorithmic'
capacity_t2['merge_applied'] = False
capacity_t2 = capacity_t2[[
    'lens', 'camp_id', 'camp_name_draft', 'countries', 'country_share', 'raw_total_weight', 'raw_weight_share',
    'dominant_mode', 'dominant_mode_share', 'second_mode', 'second_mode_share', 'top3_modes', 'top3_share',
    'dominant_status', 'dominant_tier', 'interpretable', 'source_communities', 'compression_layer', 'merge_applied'
]]

count_crosswalk_enriched = p6_crosswalk.merge(
    p3_count_profiles[['community', 'raw_total_weight', 'raw_weight_share', 'top3_share', 'dominant_status', 'dominant_tier', 'interpretable']].rename(columns={'community': 'original_count_community'}),
    on='original_count_community',
    how='left'
)
count_t2 = count_crosswalk_enriched.groupby([
    'main_text_count_camp', 'main_text_count_camp_name_draft', 'camp_group_members', 'camp_dominant_mode',
    'camp_dominant_mode_share', 'camp_top3_modes'
], as_index=False).agg(
    countries=('original_countries', 'sum'),
    raw_total_weight=('raw_total_weight', 'sum'),
    raw_weight_share=('raw_weight_share', 'sum'),
    top3_share=('top3_share', 'mean'),
    merge_applied=('merge_applied', 'max'),
)
count_t2['lens'] = 'count_higher_order'
count_t2['camp_id'] = count_t2['main_text_count_camp']
count_t2['camp_name_draft'] = count_t2['main_text_count_camp_name_draft']
count_t2['country_share'] = count_t2['countries'] / EXPECTED_COUNTRIES
count_t2['dominant_mode'] = count_t2['camp_dominant_mode']
count_t2['dominant_mode_share'] = count_t2['camp_dominant_mode_share']
count_t2['second_mode'] = count_t2['camp_top3_modes'].str.split(' / ').str[1].fillna('')
count_t2['second_mode_share'] = np.nan
count_t2['top3_modes'] = count_t2['camp_top3_modes']
count_t2['dominant_status'] = count_t2['dominant_mode'].str.split(' \| ').str[0]
count_t2['dominant_tier'] = count_t2['dominant_mode'].str.split(' \| ').str[1]
count_t2['interpretable'] = True
count_t2['source_communities'] = count_t2['camp_group_members']
count_t2['compression_layer'] = 'higher_order'
count_t2 = count_t2[[
    'lens', 'camp_id', 'camp_name_draft', 'countries', 'country_share', 'raw_total_weight', 'raw_weight_share',
    'dominant_mode', 'dominant_mode_share', 'second_mode', 'second_mode_share', 'top3_modes', 'top3_share',
    'dominant_status', 'dominant_tier', 'interpretable', 'source_communities', 'compression_layer', 'merge_applied'
]]

T2_source = pd.concat([capacity_t2, count_t2], ignore_index=True)
T2_source = T2_source.sort_values(['lens', 'camp_id']).reset_index(drop=True)
T2_source = with_object_metadata(T2_source, 'T2', 'table', OBJECT_TITLES['T2'], CLAIM_SUPPORT['T2'], 'P3+P6')
T2_source.to_csv(TABLES / 'p8_T2_source.csv', index=False)

# T3
null_rows = p4_summary.copy()
null_rows['summary_section'] = 'null_model'
null_rows['scenario_name'] = 'baseline_null'
null_rows['scenario_label'] = 'Degree-preserving null comparison'
null_rows['countries'] = EXPECTED_COUNTRIES
null_rows['rows'] = metric_value('rows_retained', int)
null_rows['capacity_mw'] = metric_value('capacity_mw_retained', float)
null_rows['ari_vs_baseline'] = np.nan
null_rows['status_stability_top20'] = np.nan
null_rows['tier_stability_top20'] = np.nan
null_rows['dominant_profile_stability_top20'] = np.nan
null_rows['camp_signature_overlap_top5'] = np.nan
null_rows['notes'] = 'Observed-vs-null summary under the frozen baseline projection.'
null_rows = null_rows[[
    'summary_section', 'scenario_name', 'scenario_label', 'weight_kind', 'countries', 'rows', 'capacity_mw',
    'observed_modularity', 'null_mean', 'null_std', 'z_score', 'null_p_ge_observed',
    'observed_communities', 'observed_top5_strength_share', 'ari_vs_baseline', 'status_stability_top20',
    'tier_stability_top20', 'dominant_profile_stability_top20', 'camp_signature_overlap_top5', 'notes'
]]

binning_rows = p7_binning.rename(columns={'communities': 'observed_communities'}).copy()
binning_rows['summary_section'] = 'robustness_binning'
binning_rows['observed_modularity'] = binning_rows['modularity']
binning_rows['null_mean'] = np.nan
binning_rows['null_std'] = np.nan
binning_rows['z_score'] = np.nan
binning_rows['null_p_ge_observed'] = np.nan
binning_rows['observed_top5_strength_share'] = binning_rows['top5_strength_share']
binning_rows['notes'] = 'Approved alternative size-bin sensitivity.'
binning_rows = binning_rows[[
    'summary_section', 'scenario_name', 'scenario_label', 'weight_kind', 'countries', 'rows', 'capacity_mw',
    'observed_modularity', 'null_mean', 'null_std', 'z_score', 'null_p_ge_observed',
    'observed_communities', 'observed_top5_strength_share', 'ari_vs_baseline', 'status_stability_top20',
    'tier_stability_top20', 'dominant_profile_stability_top20', 'camp_signature_overlap_top5', 'notes'
]]

countryfilter_rows = p7_countryfilter.rename(columns={
    'filter_name': 'scenario_name',
    'filter_label': 'scenario_label',
    'communities': 'observed_communities',
    'top5_strength_share': 'observed_top5_strength_share',
    'ari_vs_baseline_on_common': 'ari_vs_baseline',
    'dominant_mode_stability_top20': 'dominant_profile_stability_top20',
}).copy()
countryfilter_rows['summary_section'] = 'robustness_country_filter'
countryfilter_rows['observed_modularity'] = countryfilter_rows['modularity']
countryfilter_rows['null_mean'] = np.nan
countryfilter_rows['null_std'] = np.nan
countryfilter_rows['z_score'] = np.nan
countryfilter_rows['null_p_ge_observed'] = np.nan
countryfilter_rows['notes'] = 'Optional capacity-based country-filter sensitivity; descriptive support only.'
countryfilter_rows = countryfilter_rows[[
    'summary_section', 'scenario_name', 'scenario_label', 'weight_kind', 'countries', 'rows', 'capacity_mw',
    'observed_modularity', 'null_mean', 'null_std', 'z_score', 'null_p_ge_observed',
    'observed_communities', 'observed_top5_strength_share', 'ari_vs_baseline', 'status_stability_top20',
    'tier_stability_top20', 'dominant_profile_stability_top20', 'camp_signature_overlap_top5', 'notes'
]]

T3_source = pd.concat([null_rows, binning_rows, countryfilter_rows], ignore_index=True)
T3_source = with_object_metadata(T3_source, 'T3', 'table', OBJECT_TITLES['T3'], CLAIM_SUPPORT['T3'], 'P4+P7')
T3_source.to_csv(TABLES / 'p8_T3_source.csv', index=False)

display(Markdown('**Table source files saved**'))
display(pd.DataFrame({'object_code': EXPECTED_MAIN_TABLES, 'title': [OBJECT_TITLES[x] for x in EXPECTED_MAIN_TABLES]}))



## Object QA and freeze report


In [ ]:

qa_rows = []

qa_rows.append({'check': 'figure_count', 'passed': len(EXPECTED_MAIN_FIGURES) == 6, 'detail': f'{len(EXPECTED_MAIN_FIGURES)} main figures frozen'})
qa_rows.append({'check': 'table_count', 'passed': len(EXPECTED_MAIN_TABLES) == 3, 'detail': f'{len(EXPECTED_MAIN_TABLES)} main tables frozen'})
qa_rows.append({'check': 'F3_country_rows', 'passed': len(F3_source) == EXPECTED_COUNTRIES, 'detail': f'F3 rows={len(F3_source)}'})
qa_rows.append({'check': 'F3_communities', 'passed': F3_source['community'].nunique() == EXPECTED_CAPACITY_COMMUNITIES, 'detail': f'F3 communities={F3_source["community"].nunique()}'})
qa_rows.append({'check': 'F5_country_rows', 'passed': len(F5_source) == EXPECTED_COUNTRIES, 'detail': f'F5 rows={len(F5_source)}'})
qa_rows.append({'check': 'F5_original_communities', 'passed': F5_source['community'].nunique() == EXPECTED_COUNT_ORIGINAL_COMMUNITIES, 'detail': f'F5 original communities={F5_source["community"].nunique()}'})
qa_rows.append({'check': 'F5_main_text_camps', 'passed': F5_source['main_text_count_camp'].nunique() == EXPECTED_COUNT_MAINTEXT_CAMPS, 'detail': f'F5 main-text camps={F5_source["main_text_count_camp"].nunique()}'})
qa_rows.append({'check': 'F6_mode_rows', 'passed': len(F6_source) == EXPECTED_MODES, 'detail': f'F6 mode rows={len(F6_source)}'})
qa_rows.append({'check': 'T2_capacity_rows', 'passed': int((T2_source['lens'] == 'capacity_main').sum()) == EXPECTED_CAPACITY_COMMUNITIES, 'detail': f'T2 capacity rows={(T2_source["lens"] == "capacity_main").sum()}'})
qa_rows.append({'check': 'T2_count_rows', 'passed': int((T2_source['lens'] == 'count_higher_order').sum()) == EXPECTED_COUNT_MAINTEXT_CAMPS, 'detail': f'T2 count rows={(T2_source["lens"] == "count_higher_order").sum()}'})
qa_rows.append({'check': 'T3_rows', 'passed': len(T3_source) == 8, 'detail': f'T3 rows={len(T3_source)}'})
qa_rows.append({'check': 'required_files_exist', 'passed': all((TABLES / f'p8_{code}_source.csv').exists() for code in EXPECTED_MAIN_FIGURES + EXPECTED_MAIN_TABLES), 'detail': 'all 9 required source files saved'})

qa_df = pd.DataFrame(qa_rows)
if not qa_df['passed'].all():
    raise ValueError('P8 object freeze QA failed. Inspect the QA summary table.')

object_inventory = pd.DataFrame([
    {'object_code': code, 'object_type': 'figure' if code.startswith('F') else 'table', 'title': OBJECT_TITLES[code], 'claim_support': CLAIM_SUPPORT[code]}
    for code in EXPECTED_MAIN_FIGURES + EXPECTED_MAIN_TABLES
])

report_lines = [
    '# P8 Figure/Table Freeze',
    '',
    '## Run metadata',
    f'- run_id: `{RUN_ID}`',
    f'- pipeline_version: `{PIPELINE_VERSION}`',
    '- role: manuscript-facing source assembly only; no analytical reruns',
    '',
    '## Frozen main figure set',
]
for code in EXPECTED_MAIN_FIGURES:
    report_lines.append(f'- {code}: {OBJECT_TITLES[code]} [{CLAIM_SUPPORT[code]}]')
report_lines.extend([
    '',
    '## Frozen main table set',
])
for code in EXPECTED_MAIN_TABLES:
    report_lines.append(f'- {code}: {OBJECT_TITLES[code]} [{CLAIM_SUPPORT[code]}]')
report_lines.extend([
    '',
    '## Upstream routing decisions',
    '- F1 draws workflow + retained-scope context from P1-P7 and anchors the retained analytical frame.',
    '- F2 is sourced directly from the compact P2 concentration diagnostics rather than from the full matrices.',
    '- F3 keeps the country-level capacity camp membership layer with merged community profile metadata.',
    '- F4 keeps the full null-draw distribution plus observed summary in one source table.',
    '- F5 keeps country-level count membership joined to the P6 higher-order camp crosswalk.',
    '- F6 keeps the cross-view mode distinction comparison table rather than the full leave-one-mode-out diagnostics.',
    '- T1 stays compact and scope-defining; country-level dumps remain outside the main table.',
    '- T2 combines 6 capacity camps and 5 higher-order count camps in one manuscript summary object.',
    '- T3 combines null-model and robustness summaries; full null draws and rerun details stay outside the main table.',
    '',
    '## SI / external routing notes',
    '- Full matrices, full memberships, full null draws, and full sensitivity reruns remain SI/external material rather than main-text tables.',
    '- `p7_altlow_community_outputs.csv` and `p7_altgiga_community_outputs.csv` are preserved as robustness-facing supporting files, not main-text tables.',
    '- `p6_count_community_to_camp_crosswalk.csv` remains the authoritative SI.3 crosswalk, while F5/T2 use its compressed main-text layer.',
    '',
    '## QA summary',
])
for row in qa_df.itertuples(index=False):
    report_lines.append(f'- {row.check}: passed={bool(row.passed)}; {row.detail}')
report_lines.extend([
    '',
    '## Output inventory',
])
for code in EXPECTED_MAIN_FIGURES + EXPECTED_MAIN_TABLES:
    report_lines.append(f'- `artifacts/tables/p8_{code}_source.csv`')
report_lines.extend([
    '- `artifacts/reports/p8_figure_table_freeze.md`',
    '- `artifacts/runtime/p8_source_assembly_manifest.json`',
])

(REPORTS / 'p8_figure_table_freeze.md').write_text('\n'.join(report_lines), encoding='utf-8')

display(Markdown((REPORTS / 'p8_figure_table_freeze.md').read_text(encoding='utf-8')))
display(Markdown('**QA summary**'))
display(qa_df)
display(Markdown('**Object inventory**'))
display(object_inventory)
